In [12]:
import numpy as np
import zarr
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")   # safer non-interactive backend
import matplotlib.pyplot as plt
import random

In [8]:
ZARR_DISP   = "displacements.zarr"
ZARR_SDV    = "ip_nodal.zarr"

R_MODES = 50   # POD modes used for latent codes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [9]:
class LatentGRUWithParams(nn.Module):
    def __init__(self, input_dim=150+8, hidden_dim=256, num_layers=2, output_dim=150):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None):
        # x: (B,T,input_dim)
        h_out, h_n = self.gru(x, h0)   # (B,T,H)
        y = self.linear(h_out)         # (B,T,output_dim)
        return y, h_n


In [13]:
all_sim_ids = sorted(set(int(s) for s in sim_index))

# filter sims that actually have at least 2 frames
sims_valid = []
for sim in all_sim_ids:
    idx = np.where(sim_index == sim)[0]
    if idx.size >= 2:
        sims_valid.append(sim)

print("Total sims with T>=2:", len(sims_valid))

# Reproduce the 80/20 split (same seed as training)
random.seed(42)
random.shuffle(sims_valid)

split = int(0.8 * len(sims_valid))
train_sims = sims_valid[:split]
val_sims   = sims_valid[split:]

print("Approx train sims:", len(train_sims), "val sims:", len(val_sims))


Total sims with T>=2: 927
Approx train sims: 741 val sims: 186


In [15]:
def get_normalized_latent_sequence_for_sim(sim_id: int):
    """
    Returns:
      seq_lat_norm: (T_i, 150) normalized latents for a single simulation.
    """
    # indices of snapshots belonging to this sim
    idx = np.where(sim_index == sim_id)[0]
    if idx.size < 2:
        return None   # too short

    # sort by frame index to ensure time order
    idx_sorted = idx[np.argsort(frame_index[idx])]
    T = idx_sorted.size

    # pull u, g1, g2 for these snapshot indices only
    # U_lat_z is (50, S) → select columns idx_sorted → (50, T) then transpose
    U_seq  = np.asarray(U_lat_z[:,  idx_sorted], dtype=np.float32).T  # (T,50)
    G1_seq = np.asarray(G1_lat_z[:, idx_sorted], dtype=np.float32).T  # (T,50)
    G2_seq = np.asarray(G2_lat_z[:, idx_sorted], dtype=np.float32).T  # (T,50)

    # combine to (T,150)
    X_seq = np.concatenate([U_seq, G1_seq, G2_seq], axis=1)           # (T,150)

    # normalize with stored stats
    X_seq_norm = (X_seq - lat_mean[None, :]) / lat_std[None, :]

    return X_seq_norm


In [17]:
# ---- open zarr groups (read-only is fine) ----
g_disp = zarr.open_group(ZARR_DISP, mode="r")
g_sdv  = zarr.open_group(ZARR_SDV,  mode="r")

# latent coeffs: shape (50, S), small
U_lat_z  = g_disp["latent_displ_r50"]["coeffs"]   # zarr array (50, S)
G1_lat_z = g_sdv["latent_SDV1_r50"]["coeffs"]    # (50, S)
G2_lat_z = g_sdv["latent_SDV4_r50"]["coeffs"]    # (50, S)

# mapping arrays (small)
sim_index   = g_disp["mapping"]["sim_index"][:]    # (S,)
frame_index = g_disp["mapping"]["frame_index"][:]  # (S,)

S = sim_index.shape[0]
print("Total snapshots S =", S)

# latent normalization stats (tiny)
norm_grp = g_disp["latent_norm_stats"]
lat_mean = norm_grp["mean"][:].astype(np.float32)   # (150,)
lat_std  = norm_grp["std"][:].astype(np.float32)    # (150,)

print("lat_mean shape:", lat_mean.shape, "lat_std shape:", lat_std.shape)

# design parameters + norm stats (tiny)
design_raw = np.loadtxt(DESIGN_TXT)
sim_ids_design = design_raw[:, 0].astype(int)    # 1..N
params_raw     = design_raw[:, 1:].astype(np.float32)  # (N,8)

assert np.all(sim_ids_design == np.arange(1, len(sim_ids_design)+1)), "Design sim IDs not 1..N"

n_sims = len(sim_ids_design)
params_by_sim = params_raw                        # (N,8)

param_grp = g_disp["param_norm_stats"]
p_mean = param_grp["mean"][:].astype(np.float32)  # (8,)
p_std  = param_grp["std"][:].astype(np.float32)   # (8,)

params_norm_by_sim = (params_by_sim - p_mean[None, :]) / p_std[None, :]
print("params_norm_by_sim shape:", params_norm_by_sim.shape)


Total snapshots S = 42237
lat_mean shape: (150,) lat_std shape: (150,)


NameError: name 'DESIGN_TXT' is not defined

In [16]:
val_errors = []
val_sims_used = []   # in case any sims are skipped for having <2 steps after slicing

with torch.no_grad():
    for sim in val_sims:
        seq_lat = get_normalized_latent_sequence_for_sim(sim)   # (T,150) or None
        if seq_lat is None or seq_lat.shape[0] < 2:
            continue

        T = seq_lat.shape[0]
        x_lat = seq_lat[:-1, :]   # (T-1,150)
        y_lat = seq_lat[1:, :]    # (T-1,150)

        # normalized params for this sim:
        # design file is 1-based sim_id, so index sim-1
        p = params_norm_by_sim[sim-1]          # (8,)
        p_tiled = np.repeat(p[None, :], T-1, axis=0)   # (T-1,8)

        x_full = np.concatenate([x_lat, p_tiled], axis=1)   # (T-1,158)

        x_t = torch.tensor(x_full, dtype=torch.float32, device=device).unsqueeze(0)  # (1,T-1,158)
        y_t = torch.tensor(y_lat,  dtype=torch.float32, device=device).unsqueeze(0)  # (1,T-1,150)

        y_pred, _ = model_eval(x_t)   # (1,T-1,150)

        mse = torch.mean((y_pred - y_t)**2).item()
        val_errors.append(mse)
        val_sims_used.append(sim)

val_errors = np.array(val_errors)
val_sims_used = np.array(val_sims_used, dtype=int)

print("Val sims used:", len(val_sims_used))
print("Val MSE: min={:.4e}, median={:.4e}, max={:.4e}".format(
    val_errors.min(), np.median(val_errors), val_errors.max()
))


NameError: name 'U_lat_z' is not defined

In [3]:
# --- open groups ---
g_disp = zarr.open_group(ZARR_DISP, mode="r")
g_sdv  = zarr.open_group(ZARR_SDV,  mode="r")

# --- mapping (shared for all fields) ---
sim_index   = g_disp["mapping"]["sim_index"][:]    # (S,)
frame_index = g_disp["mapping"]["frame_index"][:]  # (S,)
time_vals   = g_disp["mapping"]["time_vals"][:]    # (S,)
S = sim_index.shape[0]
print("Total snapshots S =", S)

# --- displacement POD basis & mean ---
U_u   = np.asarray(g_disp["pod_full"]["U"][:, :R_MODES], dtype=np.float64)   # (M_u, r)
mu_u  = g_disp["pod_full"]["mean"][:].astype(np.float64)                     # (M_u,)
M_u   = U_u.shape[0]
print("U_u shape:", U_u.shape, "mu_u:", mu_u.shape)

# --- displacement latent coeffs ---
A_u_z = g_disp["latent_displ_r50"]["coeffs"]   # zarr array (r, S), float32

# --- SDV1 POD basis & mean ---
U_g1  = np.asarray(g_sdv["SDV1_pod_full"]["U_SDV1"][:, :R_MODES], dtype=np.float64)  # (M_g1, r)
mu_g1 = g_sdv["SDV1_pod_full"]["mean_SDV1"][:].astype(np.float64)                    # (M_g1,)
M_g1  = U_g1.shape[0]
print("U_g1 shape:", U_g1.shape, "mu_g1:", mu_g1.shape)

# --- SDV1 latent coeffs ---
A_g1_z = g_sdv["latent_SDV1_r50"]["coeffs"]   # (r, S)

# --- SDV4 POD basis & mean ---
U_g2  = np.asarray(g_sdv["SDV4_pod_full"]["U_SDV4"][:, :R_MODES], dtype=np.float64)  # (M_g2, r)
mu_g2 = g_sdv["SDV4_pod_full"]["mean_SDV4"][:].astype(np.float64)                    # (M_g2,)
M_g2  = U_g2.shape[0]
print("U_g2 shape:", U_g2.shape, "mu_g2:", mu_g2.shape)

# --- SDV4 latent coeffs ---
A_g2_z = g_sdv["latent_SDV4_r50"]["coeffs"]   # (r, S)

# --- original snapshots (we will only slice columns we need) ---
X_u_z   = g_disp["snapshots"]     # (M_u, S)
X_g1_z  = g_sdv["snaps_SDV1"]     # (M_g1, S)
X_g2_z  = g_sdv["snaps_SDV4"]     # (M_g2, S)


Total snapshots S = 42237
U_u shape: (11163, 50) mu_u: (11163,)
U_g1 shape: (11163, 50) mu_g1: (11163,)
U_g2 shape: (11163, 50) mu_g2: (11163,)


In [4]:
def get_sim_indices(sim_id: int):
    """Return sorted indices and times for a given simulation ID."""
    idx = np.where(sim_index == sim_id)[0]
    if idx.size == 0:
        raise ValueError(f"No snapshots found for sim_id={sim_id}")
    idx_sorted = idx[np.argsort(frame_index[idx])]
    t_sim = time_vals[idx_sorted]   # (T,)
    return idx_sorted, t_sim


In [5]:
def reconstruct_and_mse_for_sim(sim_id, U, mu, A_z, X_z, field_name="field"):
    """
    Returns:
      times   : (T,)
      mse_vec : (T,) MSE per snapshot
      X_true  : (M,T) original snapshots
      X_rec   : (M,T) reconstructed snapshots
    """
    idx_sorted, t_sim = get_sim_indices(sim_id)
    T = idx_sorted.size

    print(f"[{field_name}] sim_id={sim_id}, T={T}")

    # --- load latent coeffs for this sim: (r,T) ---
    A_sim = np.asarray(A_z[:, idx_sorted], dtype=np.float64)   # (r,T)

    # --- reconstruct: X_rec = mu[:,None] + U @ A_sim ---
    X_rec = mu[:, None] + U @ A_sim                            # (M,T)

    # --- load true snapshots for this sim ---
    X_true = np.asarray(X_z[:, idx_sorted], dtype=np.float64)  # (M,T)

    # --- MSE over spatial DOFs for each time ---
    diff = X_rec - X_true
    mse_vec = np.mean(diff*diff, axis=0)                       # (T,)

    print(f"[{field_name}] MSE: min={mse_vec.min():.3e}, max={mse_vec.max():.3e}")

    return t_sim, mse_vec, X_true, X_rec


In [6]:
print("Best sim:", best_sim)
print("Median sim:", median_sim)
print("Worst sim:", worst_sim)


NameError: name 'best_sim' is not defined

In [ ]:
sims_to_do = {
    "best":   best_sim,
    "median": median_sim,
    "worst":  worst_sim,
}

results = {}  # store stuff for later plotting of full fields

for tag, sid in sims_to_do.items():
    print("\n======", tag.upper(), "sim_id =", sid, "======")

    # displacement u
    t_u, mse_u, X_u_true, X_u_rec = reconstruct_and_mse_for_sim(
        sid, U_u, mu_u, A_u_z, X_u_z, field_name=f"u_{tag}"
    )

    # SDV1 (lambda_g1)
    t_g1, mse_g1, X_g1_true, X_g1_rec = reconstruct_and_mse_for_sim(
        sid, U_g1, mu_g1, A_g1_z, X_g1_z, field_name=f"SDV1_{tag}"
    )

    # SDV4 (lambda_g2)
    t_g2, mse_g2, X_g2_true, X_g2_rec = reconstruct_and_mse_for_sim(
        sid, U_g2, mu_g2, A_g2_z, X_g2_z, field_name=f"SDV4_{tag}"
    )

    # sanity check: times should match across fields
    assert np.allclose(t_u, t_g1) and np.allclose(t_u, t_g2)

    results[tag] = {
        "sim_id": sid,
        "t": t_u,
        "mse_u": mse_u,
        "mse_g1": mse_g1,
        "mse_g2": mse_g2,
        "X_u_true": X_u_true,
        "X_u_rec":  X_u_rec,
        "X_g1_true": X_g1_true,
        "X_g1_rec":  X_g1_rec,
        "X_g2_true": X_g2_true,
        "X_g2_rec":  X_g2_rec,
    }


In [ ]:
for tag, data in results.items():
    t      = data["t"]
    mse_u  = data["mse_u"]
    mse_g1 = data["mse_g1"]
    mse_g2 = data["mse_g2"]

    plt.figure(figsize=(6,4))
    plt.plot(t, mse_u,  label="u")
    plt.plot(t, mse_g1, label="lambda_g1 (SDV1)")
    plt.plot(t, mse_g2, label="lambda_g2 (SDV4)")
    plt.xlabel("time")
    plt.ylabel("MSE over DOFs")
    plt.title(f"MSE vs time ({tag} sim_id={data['sim_id']})")
    plt.grid(True)
    plt.legend()
    fname = f"mse_vs_time_{tag}_sim{data['sim_id']}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved", fname)


In [ ]:
def choose_sample_indices(t):
    """Return a few representative time indices (start, mid, end)."""
    T = t.size
    idxs = [0, T//2, T-1] if T >= 3 else list(range(T))
    return idxs


In [ ]:
def displacement_magnitude_field(x_vec):
    """
    x_vec: (M_u,) with [Ux(0..N-1), Uy(0..N-1), Uz(0..N-1)]
    returns: (N,) disp magnitude
    """
    N = x_vec.shape[0] // 3
    Ux = x_vec[0:N]
    Uy = x_vec[N:2*N]
    Uz = x_vec[2*N:3*N]
    return np.sqrt(Ux*Ux + Uy*Uy + Uz*Uz)


In [ ]:
def plot_full_field_1d(field_true, field_rec, title, fname):
    """
    Simple 1D plot to compare true vs recon as a function of DOF index.
    For a nicer plot, replace with your own mesh/surface visualization.
    """
    plt.figure(figsize=(6,4))
    idx = np.arange(field_true.size)
    plt.plot(idx, field_true, label="true", alpha=0.7)
    plt.plot(idx, field_rec, "--", label="recon", alpha=0.7)
    plt.xlabel("DOF index")
    plt.ylabel("value")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved", fname)


In [ ]:
for tag, data in results.items():
    sid   = data["sim_id"]
    t     = data["t"]
    idxs  = choose_sample_indices(t)

    X_u_true  = data["X_u_true"]   # (M_u,T)
    X_u_rec   = data["X_u_rec"]    # (M_u,T)
    X_g1_true = data["X_g1_true"]  # (M_g1,T)
    X_g1_rec  = data["X_g1_rec"]
    X_g2_true = data["X_g2_true"]  # (M_g2,T)
    X_g2_rec  = data["X_g2_rec"]

    for k in idxs:
        t_k = t[k]

        # --- displacement magnitude field ---
        u_true_k = displacement_magnitude_field(X_u_true[:, k])
        u_rec_k  = displacement_magnitude_field(X_u_rec[:, k])
        title_u  = f"u magnitude ({tag}, sim={sid}, t={t_k:.3f})"
        fname_u  = f"field_u_mag_{tag}_sim{sid}_tidx{k}.png"
        plot_full_field_1d(u_true_k, u_rec_k, title_u, fname_u)

        # --- lambda_g1 (SDV1) ---
        g1_true_k = X_g1_true[:, k]
        g1_rec_k  = X_g1_rec[:, k]
        title_g1  = f"lambda_g1 SDV1 ({tag}, sim={sid}, t={t_k:.3f})"
        fname_g1  = f"field_g1_{tag}_sim{sid}_tidx{k}.png"
        plot_full_field_1d(g1_true_k, g1_rec_k, title_g1, fname_g1)

        # --- lambda_g2 (SDV4) ---
        g2_true_k = X_g2_true[:, k]
        g2_rec_k  = X_g2_rec[:, k]
        title_g2  = f"lambda_g2 SDV4 ({tag}, sim={sid}, t={t_k:.3f})"
        fname_g2  = f"field_g2_{tag}_sim{sid}_tidx{k}.png"
        plot_full_field_1d(g2_true_k, g2_rec_k, title_g2, fname_g2)
